In [24]:
import numpy as np
from PIL import Image

# загрузка изображения
im = Image.open("im1.jpg")
im = im.convert("RGB")
pixels = np.array(im, dtype=float)

h, w, c = pixels.shape

# --- 2D FFT ---
F = np.zeros((h, w, c), dtype=complex)

for k in range(c):
    # сначала по x
    temp = np.fft.fft(pixels[:, :, k], axis=1)
    # затем по y
    F[:, :, k] = np.fft.fft(temp, axis=0)

# --- сохраняем фурье-образ ---

mag = np.abs(F)
mag_log = np.log(mag + 1)  # добавляем 1, чтобы избежать log(0)
mag_log = mag_log / mag_log.max() * 255
mag = mag.astype(np.uint8)
mag_log = mag_log.astype(np.uint8)
Image.fromarray(mag).save("out.jpg")
Image.fromarray(mag_log).save("out_log.jpg")

# --- удаляем верхнюю половину частот ---
F2 = F.copy()

F2[h//2:, :, :] = 0
F2[:, w//2:, :] = 0

# --- обратное преобразование ---
restored = np.zeros((h, w, c), dtype=float)

for k in range(c):
    temp = np.fft.ifft(F2[:, :, k], axis=0)
    temp = np.fft.ifft(temp, axis=1)
    restored[:, :, k] = np.real(temp)

# ограничиваем значения
restored = np.clip(restored, 0, 255)
restored = np.round(restored).astype(np.uint8)

Image.fromarray(restored).save("out2.jpg")

(218, 217, 251)
